## Example 1: Basic HTTP Request and Response Handling

In [ ]:
import webob
from webob import Request, Response

# Example 1: Basic HTTP request and response handling
print("Basic HTTP request and response handling:")

# Create a simple GET request
get_request = Request.blank('/api/users')
print(f"GET request: {get_request.method} {get_request.path}")
print(f"Host: {get_request.host}")
print(f"URL: {get_request.url}")

# Create a POST request with data
post_request = Request.blank('/api/users', method='POST')
post_request.body = b'{"name": "Alice", "email": "alice@example.com"}'
post_request.content_type = 'application/json'
print(f"\nPOST request: {post_request.method} {post_request.path}")
print(f"Content-Type: {post_request.content_type}")
print(f"Body: {post_request.body.decode()}")

# Create various HTTP responses (with charset parameter for text bodies)
responses = [
    Response('Success!', status=200, charset='utf-8'),
    Response('Resource created', status=201, charset='utf-8'),
    Response('Not Found', status=404, charset='utf-8'),
    Response('{"error": "Server Error"}', status=500, content_type='application/json; charset=utf-8')
]

print(f"\nHTTP responses:")
for resp in responses:
    print(f"Status: {resp.status}, Content-Type: {resp.content_type}")
    print(f"Body: {resp.text[:50]}...")

# Test request properties
print(f"\nRequest properties:")
test_req = Request.blank('/search?q=python&sort=date')
print(f"Query string: {test_req.query_string}")
print(f"GET params: {dict(test_req.GET)}")
print(f"Path info: {test_req.path_info}")
print(f"Script name: {test_req.script_name}")

# Test different HTTP methods
print(f"\nHTTP methods:")
methods = ['GET', 'POST', 'PUT', 'DELETE', 'PATCH']
for method in methods:
    req = Request.blank('/api/resource', method=method)
    print(f"{method}: {req.method} {req.path}")

# Content negotiation
print(f"\nContent negotiation:")
accept_request = Request.blank('/api/data')
accept_request.accept = 'application/json, text/html;q=0.9, */*;q=0.8'
print(f"Accept header: {accept_request.accept}")
print(f"Best match for json: {accept_request.accept.best_match(['application/json', 'text/html'])}")

print(f"\nWebOb provides clean HTTP request/response abstraction")
print("Essential for WSGI applications and web frameworks")

## Example 2: Form Handling and File Uploads

In [ ]:
import webob
from webob import Request, Response
import json

# Example 2: Form handling and file uploads
print("Form handling and file uploads:")

# Simulate HTML form submission
form_data = "name=John+Doe&email=john%40example.com&age=30&subscribe=on"
form_request = Request.blank('/submit', method='POST')
form_request.body = form_data.encode()
form_request.content_type = 'application/x-www-form-urlencoded'

print(f"Form submission:")
print(f"Content-Type: {form_request.content_type}")
print(f"Raw body: {form_request.body.decode()}")
print(f"Parsed POST data: {dict(form_request.POST)}")

# Access individual form fields
print(f"Form fields:")
for key, value in form_request.POST.items():
    print(f"  {key}: {value}")

# Simulate multipart form with file upload
multipart_body = (
    b'--boundary123\r\n'
    b'Content-Disposition: form-data; name="username"\r\n\r\n'
    b'testuser\r\n'
    b'--boundary123\r\n'
    b'Content-Disposition: form-data; name="file"; filename="test.txt"\r\n'
    b'Content-Type: text/plain\r\n\r\n'
    b'This is test file content\r\n'
    b'--boundary123--\r\n'
)

upload_request = Request.blank('/upload', method='POST')
upload_request.body = multipart_body
upload_request.content_type = 'multipart/form-data; boundary=boundary123'

print(f"\nFile upload simulation:")
print(f"Content-Type: {upload_request.content_type}")
print(f"Body length: {len(upload_request.body)} bytes")

# Parse multipart data
if upload_request.POST:
    print(f"POST data available: {list(upload_request.POST.keys())}")

# Cookie handling
print(f"\nCookie handling:")
cookie_request = Request.blank('/dashboard')
cookie_request.headers['Cookie'] = 'session_id=abc123; theme=dark; lang=en'

print(f"Cookies: {dict(cookie_request.cookies)}")

# Create response with cookies
cookie_response = Response('Welcome back!')
cookie_response.set_cookie('user_id', '12345', max_age=3600)
cookie_response.set_cookie('preferences', json.dumps({'theme': 'dark'}))

print(f"Response headers: {dict(cookie_response.headers)}")

# Headers manipulation
print(f"\nHeaders manipulation:")
api_request = Request.blank('/api/data')
api_request.headers['Authorization'] = 'Bearer token123'
api_request.headers['User-Agent'] = 'MyApp/1.0'
api_request.headers['Accept'] = 'application/json'

print(f"Request headers:")
for key, value in api_request.headers.items():
    print(f"  {key}: {value}")

print(f"\nWebOb simplifies complex HTTP data handling")
print("Provides elegant APIs for forms, files, cookies, and headers")

## Example 3: WSGI Application Development Patterns

In [ ]:
import webob
from webob import Request, Response
from webob.exc import HTTPNotFound, HTTPBadRequest, HTTPInternalServerError
import json

# Example 3: WSGI application development patterns
print("WSGI application development patterns:")

class SimpleWSGIApp:
    """Simple WSGI application using WebOb"""
    
    def __init__(self):
        self.routes = {}
        self.middleware = []
    
    def route(self, path, methods=['GET']):
        """Decorator to register routes"""
        def decorator(func):
            self.routes[path] = {'handler': func, 'methods': methods}
            return func
        return decorator
    
    def __call__(self, environ, start_response):
        """WSGI application entry point"""
        request = Request(environ)
        
        try:
            response = self.handle_request(request)
        except Exception as e:
            response = HTTPInternalServerError(f"Server Error: {str(e)}")
        
        return response(environ, start_response)
    
    def handle_request(self, request):
        """Handle incoming request"""
        path = request.path_info
        method = request.method
        
        # Find matching route
        if path in self.routes:
            route_info = self.routes[path]
            if method in route_info['methods']:
                handler = route_info['handler']
                return handler(request)
            else:
                return Response(f"Method {method} not allowed", status=405)
        else:
            return HTTPNotFound(f"Path {path} not found")

# Create WSGI app instance
app = SimpleWSGIApp()

# Register routes
@app.route('/', methods=['GET'])
def home(request):
    return Response('Welcome to the WebOb WSGI app!', content_type='text/plain')

@app.route('/api/users', methods=['GET', 'POST'])
def users_api(request):
    if request.method == 'GET':
        users = [{'id': 1, 'name': 'Alice'}, {'id': 2, 'name': 'Bob'}]
        # A text body needs a charset: WebOb adds one by default only for text/* types, not application/json.
        return Response(json.dumps(users), content_type='application/json', charset='utf-8')
    elif request.method == 'POST':
        try:
            data = json.loads(request.body.decode())
            new_user = {'id': 3, 'name': data.get('name', 'Unknown')}
            return Response(json.dumps(new_user), status=201, content_type='application/json', charset='utf-8')
        except json.JSONDecodeError:
            return HTTPBadRequest("Invalid JSON")

@app.route('/api/users/<user_id>', methods=['GET'])
def user_detail(request):
    # Simulate user lookup (path parameter would need URL routing)
    user_id = request.path_info.split('/')[-1]
    if user_id.isdigit():
        user = {'id': int(user_id), 'name': f'User {user_id}'}
        return Response(json.dumps(user), content_type='application/json', charset='utf-8')
    else:
        return HTTPNotFound("User not found")

# Test the WSGI application
print("Testing WSGI application:")

# Simulate different requests
test_requests = [
    Request.blank('/'),
    Request.blank('/api/users'),
    Request.blank('/api/users', method='POST', body=b'{"name": "Charlie"}', 
                  content_type='application/json'),
    Request.blank('/nonexistent'),
    Request.blank('/api/users/123')
]

observed = {}
for req in test_requests:
    print(f"\n{req.method} {req.path_info}")
    
    # Simulate WSGI environ
    environ = req.environ
    responses = []
    
    def start_response(status, headers):
        responses.append((status, headers))
    
    # Call WSGI app
    try:
        response_iter = app(environ, start_response)
        response_body = b''.join(response_iter)
        status, headers = responses[0] if responses else ('500 Internal Server Error', [])
        observed[(req.method, req.path_info)] = (status, response_body)
        
        print(f"  Status: {status}")
        print(f"  Body: {response_body.decode()[:100]}...")
    except Exception as e:
        print(f"  Error: {e}")
        raise

# The app maps ANY handler exception to a 500 (the WSGI pattern above), so a broken route shows up
# here as a wrong status rather than as a traceback.
expected_status = {
    ('GET', '/'): '200 OK',
    ('GET', '/api/users'): '200 OK',
    ('POST', '/api/users'): '201 Created',
    ('GET', '/nonexistent'): '404 Not Found',
    ('GET', '/api/users/123'): '404 Not Found',  # '<user_id>' is a literal route key here; no URL routing
}
assert {key: value[0] for key, value in observed.items()} == expected_status, observed
assert json.loads(observed[('GET', '/api/users')][1]) == [{'id': 1, 'name': 'Alice'}, {'id': 2, 'name': 'Bob'}]
assert json.loads(observed[('POST', '/api/users')][1]) == {'id': 3, 'name': 'Charlie'}

# HTTP exception handling
print(f"\nHTTP exception examples:")
exceptions = [
    HTTPNotFound("Resource not found"),
    HTTPBadRequest("Invalid request data"), 
    HTTPInternalServerError("Database connection failed")
]

for exc in exceptions:
    print(f"{exc.status}: {exc.detail}")

print(f"\nWebOb simplifies WSGI application development")
print("Provides clean abstractions for building web frameworks")

## Example 4: Static Files and Middleware Patterns

In [ ]:
import webob
from webob import Request, Response
from webob.static import FileApp
import tempfile
import os

# Example 4: Static file serving and middleware patterns
print("Static file serving and middleware patterns:")

# Create temporary static files for demonstration
temp_dir = tempfile.mkdtemp()
static_files = {
    'index.html': '<html><body><h1>Welcome</h1></body></html>',
    'style.css': 'body { font-family: Arial; }',
    'app.js': 'console.log("Hello from JavaScript");'
}

for filename, content in static_files.items():
    file_path = os.path.join(temp_dir, filename)
    with open(file_path, 'w') as f:
        f.write(content)

print(f"Created static files in: {temp_dir}")

# Static file application
class StaticFileServer:
    """Static file server using WebOb"""
    
    def __init__(self, directory):
        self.directory = directory
        self.file_app = FileApp(directory)
    
    def __call__(self, environ, start_response):
        request = Request(environ)
        
        # Basic security check - prevent directory traversal
        if '..' in request.path_info:
            response = Response('Forbidden', status=403)
            return response(environ, start_response)
        
        # Try to serve static file
        try:
            return self.file_app(environ, start_response)
        except OSError:
            # File not found
            response = Response('File not found', status=404)
            return response(environ, start_response)

# Test static file server
static_server = StaticFileServer(temp_dir)

print(f"\nTesting static file server:")
static_requests = [
    Request.blank('/index.html'),
    Request.blank('/style.css'),
    Request.blank('/nonexistent.txt'),
    Request.blank('/../etc/passwd')  # Security test
]

for req in static_requests:
    print(f"\nGET {req.path_info}")
    
    environ = req.environ
    responses = []
    
    def start_response(status, headers):
        responses.append((status, headers))
    
    try:
        response_iter = static_server(environ, start_response)
        response_body = b''.join(response_iter)
        status, headers = responses[0]
        
        print(f"  Status: {status}")
        print(f"  Content-Type: {dict(headers).get('Content-Type', 'Unknown')}")
        print(f"  Body length: {len(response_body)} bytes")
    except Exception as e:
        print(f"  Error: {e}")
        raise

# Middleware patterns
print(f"\nMiddleware patterns:")

class LoggingMiddleware:
    """Middleware that logs requests"""
    
    def __init__(self, app):
        self.app = app
        self.requests_logged = 0
    
    def __call__(self, environ, start_response):
        request = Request(environ)
        self.requests_logged += 1
        
        print(f"[LOG {self.requests_logged}] {request.method} {request.path_info}")
        
        # Call the wrapped application
        return self.app(environ, start_response)

class CompressionMiddleware:
    """Middleware that simulates response compression"""
    
    def __init__(self, app):
        self.app = app
    
    def __call__(self, environ, start_response):
        # Capture the response
        request = Request(environ)
        responses = []
        
        def capture_start_response(status, headers):
            responses.append((status, headers))
        
        response_iter = self.app(environ, capture_start_response)
        response_body = b''.join(response_iter)
        
        # Simulate compression (just add header)
        if len(response_body) > 100:
            status, headers = responses[0]
            headers = list(headers)
            headers.append(('Content-Encoding', 'gzip'))
            start_response(status, headers)
            print(f"  [COMPRESSION] Compressed {len(response_body)} bytes")
        else:
            status, headers = responses[0]
            start_response(status, headers)
        
        return [response_body]

# Stack middleware
wrapped_app = LoggingMiddleware(CompressionMiddleware(static_server))

print(f"\nTesting middleware stack:")
middleware_requests = [
    Request.blank('/index.html'),
    Request.blank('/style.css')
]

for req in middleware_requests:
    environ = req.environ
    responses = []
    
    def start_response(status, headers):
        responses.append((status, headers))
    
    response_iter = wrapped_app(environ, start_response)
    response_body = b''.join(response_iter)
    status, headers = responses[0]
    
    print(f"  Final status: {status}")
    headers_dict = dict(headers)
    if 'Content-Encoding' in headers_dict:
        print(f"  Compression applied: {headers_dict['Content-Encoding']}")

# Cleanup
import shutil
shutil.rmtree(temp_dir)

print(f"\nWebOb enables powerful middleware composition")
print("Perfect for building modular web application architectures")

## Example 5: Production Web Application Patterns

In [ ]:
import webob
from webob import Request, Response
from webob.exc import HTTPFound, HTTPUnauthorized, HTTPForbidden
import json
import hashlib
import time

# Example 5: Production web application patterns
class ProductionWebApp:
    """Production-ready web application using WebOb"""
    
    def __init__(self):
        self.routes = {}
        self.middleware_stack = []
        self.sessions = {}  # In-memory session store (use Redis in production)
        self.users = {  # Mock user database
            'admin': {'password_hash': self._hash_password('admin123'), 'role': 'admin'},
            'user': {'password_hash': self._hash_password('user123'), 'role': 'user'}
        }
        self.stats = {'requests': 0, 'errors': 0, 'auth_attempts': 0}
    
    def _hash_password(self, password):
        """Hash password for storage"""
        return hashlib.sha256(password.encode()).hexdigest()
    
    def route(self, path, methods=['GET'], auth_required=False):
        """Route decorator with authentication support"""
        def decorator(func):
            self.routes[path] = {
                'handler': func, 
                'methods': methods, 
                'auth_required': auth_required
            }
            return func
        return decorator
    
    def __call__(self, environ, start_response):
        """WSGI application entry point"""
        request = Request(environ)
        self.stats['requests'] += 1
        
        try:
            # Add security headers
            response = self.handle_request(request)
            response.headers['X-Content-Type-Options'] = 'nosniff'
            response.headers['X-Frame-Options'] = 'DENY'
            response.headers['X-XSS-Protection'] = '1; mode=block'
            
        except Exception as e:
            self.stats['errors'] += 1
            response = Response(
                json.dumps({'error': 'Internal server error'}),
                status=500,
                content_type='application/json; charset=utf-8'
            )
        
        return response(environ, start_response)
    
    def handle_request(self, request):
        """Handle incoming request with authentication and routing"""
        path = request.path_info
        method = request.method
        
        # Find route
        if path not in self.routes:
            return Response(
                json.dumps({'error': 'Not found'}),
                status=404,
                content_type='application/json; charset=utf-8'
            )
        
        route_info = self.routes[path]
        
        # Check method
        if method not in route_info['methods']:
            return Response(
                json.dumps({'error': f'Method {method} not allowed'}),
                status=405,
                content_type='application/json; charset=utf-8'
            )
        
        # Check authentication
        if route_info['auth_required']:
            auth_result = self.check_authentication(request)
            if not auth_result['authenticated']:
                # The first positional argument of an HTTPException is `detail`, rendered into its
                # plain-text/HTML template; `json_body=` is what makes the response body JSON.
                return HTTPUnauthorized(json_body={'error': 'Authentication required'})
            request.user = auth_result['user']
        
        # Call handler
        return route_info['handler'](request)
    
    def check_authentication(self, request):
        """Check request authentication"""
        self.stats['auth_attempts'] += 1
        
        # Check session cookie
        session_id = request.cookies.get('session_id')
        if session_id and session_id in self.sessions:
            session = self.sessions[session_id]
            if session['expires'] > time.time():
                return {'authenticated': True, 'user': session['user']}
        
        # Check Authorization header
        auth_header = request.headers.get('Authorization', '')
        if auth_header.startswith('Bearer '):
            token = auth_header[7:]
            # Simple token validation (use JWT in production)
            if token in ['admin_token', 'user_token']:
                user = 'admin' if token == 'admin_token' else 'user'
                return {'authenticated': True, 'user': user}
        
        return {'authenticated': False, 'user': None}
    
    def create_session(self, username):
        """Create user session"""
        session_id = hashlib.md5(f"{username}{time.time()}".encode()).hexdigest()
        self.sessions[session_id] = {
            'user': username,
            'created': time.time(),
            'expires': time.time() + 3600  # 1 hour
        }
        return session_id

# Create production app
app = ProductionWebApp()

# Register routes
@app.route('/api/login', methods=['POST'])
def login(request):
    try:
        data = json.loads(request.body.decode())
        username = data.get('username')
        password = data.get('password')
        
        if username in app.users:
            user_data = app.users[username]
            password_hash = app._hash_password(password)
            
            if password_hash == user_data['password_hash']:
                session_id = app.create_session(username)
                
                response = Response(
                    json.dumps({'success': True, 'role': user_data['role']}),
                    content_type='application/json; charset=utf-8'
                )
                response.set_cookie('session_id', session_id, max_age=3600)
                return response
        
        return Response(
            json.dumps({'error': 'Invalid credentials'}),
            status=401,
            content_type='application/json; charset=utf-8'
        )
    except json.JSONDecodeError:
        return Response(
            json.dumps({'error': 'Invalid JSON'}),
            status=400,
            content_type='application/json; charset=utf-8'
        )

@app.route('/api/profile', methods=['GET'], auth_required=True)
def profile(request):
    user_data = app.users.get(request.user, {})
    profile_data = {
        'username': request.user,
        'role': user_data.get('role', 'unknown'),
        'authenticated': True
    }
    return Response(
        json.dumps(profile_data), 
        content_type='application/json; charset=utf-8'
    )

@app.route('/api/admin/stats', methods=['GET'], auth_required=True)
def admin_stats(request):
    user_data = app.users.get(request.user, {})
    if user_data.get('role') != 'admin':
        return HTTPForbidden(json_body={'error': 'Admin access required'})
    
    return Response(
        json.dumps(app.stats), 
        content_type='application/json; charset=utf-8'
    )

@app.route('/health', methods=['GET'])
def health_check(request):
    return Response(
        json.dumps({'status': 'healthy', 'timestamp': time.time()}),
        content_type='application/json; charset=utf-8'
    )

@app.route('/api/logout', methods=['POST'], auth_required=True)
def logout(request):
    session_id = request.cookies.get('session_id')
    if session_id and session_id in app.sessions:
        del app.sessions[session_id]
    
    response = Response(
        json.dumps({'success': True, 'message': 'Logged out'}),
        content_type='application/json; charset=utf-8'
    )
    response.delete_cookie('session_id')
    return response

# Test production application
print("Production web application testing:")

test_scenarios = [
    # Health check
    ('Health check', Request.blank('/health')),
    
    # Login
    ('Login with valid credentials', 
     Request.blank('/api/login', method='POST', 
                   body=json.dumps({'username': 'admin', 'password': 'admin123'}).encode(),
                   content_type='application/json')),
    
    # Profile access without auth
    ('Profile without auth', Request.blank('/api/profile')),
    
    # Profile access with token
    ('Profile with token', 
     Request.blank('/api/profile', headers={'Authorization': 'Bearer admin_token'})),
    
    # Admin stats with admin token
    ('Admin stats (admin)', 
     Request.blank('/api/admin/stats', headers={'Authorization': 'Bearer admin_token'})),
    
    # Non-admin trying admin endpoint
    ('Admin stats (user)', 
     Request.blank('/api/admin/stats', headers={'Authorization': 'Bearer user_token'})),
]

observed = {}
for name, req in test_scenarios:
    print(f"\n{name}: {req.method} {req.path_info}")
    
    environ = req.environ
    responses = []
    
    def start_response(status, headers):
        responses.append((status, headers))
    
    response_iter = app(environ, start_response)
    response_body = b''.join(response_iter)
    status, headers = responses[0]
    
    print(f"  Status: {status}")
    response_data = json.loads(response_body.decode())  # every route of this JSON API answers in JSON
    print(f"  Response: {response_data}")
    observed[name] = (status, response_data)

assert observed['Health check'][0] == '200 OK' and observed['Health check'][1]['status'] == 'healthy', observed
assert observed['Login with valid credentials'] == ('200 OK', {'success': True, 'role': 'admin'}), observed
assert observed['Profile without auth'] == ('401 Unauthorized', {'error': 'Authentication required'}), observed
assert observed['Profile with token'] == (
    '200 OK', {'username': 'admin', 'role': 'admin', 'authenticated': True}), observed
assert observed['Admin stats (admin)'][0] == '200 OK' and observed['Admin stats (admin)'][1]['errors'] == 0, observed
assert observed['Admin stats (user)'] == ('403 Forbidden', {'error': 'Admin access required'}), observed
assert app.stats['errors'] == 0, app.stats  # the app's own 500 handler never fired

print(f"\nApplication statistics: {app.stats}")

print(f"\nProduction features demonstrated:")
features = [
    "Authentication and session management",
    "Role-based access control (RBAC)",
    "Security headers (XSS, CSRF, clickjacking protection)",
    "JSON API endpoints with proper error handling",
    "Password hashing for secure storage",
    "Health checks and monitoring endpoints",
    "Performance statistics tracking",
    "Cookie-based sessions with expiration"
]

for feature in features:
    print(f"  ✓ {feature}")

print(f"\nWebOb provides enterprise-grade web application foundation")
print("Used by Pyramid and many other Python web frameworks")